# 第 5 节课 · YOLO 目标检测实战

## 本 Notebook 目标

完成本 Notebook 后，你将能够：
1. 理解目标检测和图像分类的本质区别
2. 理解 YOLO 的"一次看完全图"思想
3. 使用 Ultralytics YOLOv8 加载预训练模型做推理
4. 理解置信度阈值（conf）、NMS IoU 阈值（iou）的作用
5. 通过调参观察检测框的变化
6. 对比不同大小 YOLOv8 模型的速度与精度

## 目标检测是什么？

前面我们学的是**图像分类**：输入一张图，输出一个类别标签。

目标检测更进一步：**不仅要知道图里有什么，还要知道它们在哪里**。

输出格式通常是多个边界框（bounding box），每个框包含：
- 框的坐标：$(x_1, y_1, x_2, y_2)$ 或 $(x, y, w, h)$
- 置信度：这个框里确实有目标的概率
- 类别：目标属于哪个类别

应用场景：自动驾驶、安防监控、医学影像、工业质检、人脸识别等。


## 1. 环境准备

本节课需要安装 `ultralytics` 包，它提供了 YOLOv8 的实现。

```bash
pip install ultralytics
```

第一次运行时，YOLOv8 预训练权重会自动下载到当前目录（`yolov8n.pt` 约 6MB）。


In [ ]:
# 导入必要的库
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch

# 检查 ultralytics 是否安装
try:
    from ultralytics import YOLO
    print(f"ultralytics 已安装")
except ImportError:
    print("请先安装 ultralytics: pip install ultralytics")

# 自动选择设备
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"使用设备: {device}")


## 2. 加载 YOLOv8 预训练模型

YOLOv8 提供了多个版本的模型，主要区别是速度和精度：

| 模型 | 参数量 | 速度 | 精度 | 适用场景 |
|------|:------:|:----:|:----:|----------|
| yolov8n | 3.2M | 最快 | 较低 | 实时应用、边缘设备 |
| yolov8s | 11.2M | 快 | 中等 | 平衡选择 |
| yolov8m | 25.9M | 中等 | 较高 | 精度优先 |
| yolov8l | 43.7M | 慢 | 高 | 服务器端 |
| yolov8x | 68.2M | 最慢 | 最高 | 追求极致精度 |

本节课主要用 `yolov8n`（nano 版），因为它速度快，适合演示。


In [ ]:
# 加载 yolov8n 预训练模型
# 首次运行会自动下载 yolov8n.pt（约 6MB）
model = YOLO('yolov8n.pt')

# 查看模型支持的类别
print(f"模型类别数: {len(model.names)}")
print(f"前 10 个类别: {list(model.names.values())[:10]}")

# 查看模型参数量
print(f"模型参数量: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")


## 3. YOLO 检测流程

YOLO 的检测流程可以概括为：

```
输入图片
    ↓
骨干网络提取特征
    ↓
检测头预测：每个网格的类别、框坐标、置信度
    ↓
置信度过滤（去掉低置信度的框）
    ↓
NMS 非极大值抑制（去掉重叠框）
    ↓
输出最终检测框
```

### 核心概念

**置信度（Confidence Score）**
- 表示模型认为这个框里确实有目标，并且类别正确的程度
- 置信度 = 目标存在概率 × 类别概率
- `conf` 阈值越高，保留的框越少但越可靠

**IoU（Intersection over Union）**
- 两个框的重叠程度
- IoU = 交集面积 / 并集面积
- IoU 越大，两个框越重叠

**NMS（Non-Maximum Suppression）**
- 对同一个目标可能产生多个重叠框
- NMS 保留置信度最高的框，去掉与它 IoU 超过阈值的其他框
- `iou` 阈值越高，NMS 越宽松，保留的框越多


## 4. 单张图片推理

我们用一张示例图片做推理。如果没有本地图片，可以用 ultralytics 内置的 bus.jpg。


In [ ]:
# 准备一张测试图片
# 如果有本地图片，可以修改这个路径
image_path = 'sample.jpg'

if not os.path.exists(image_path):
    # 使用 ultralytics 内置示例图片
    # 它会在 ultralytics 包内部找到 bus.jpg
    image_path = None
    print("未找到本地 sample.jpg，使用 ultralytics 内置示例图片")
else:
    print(f"使用本地图片: {image_path}")

# 运行推理
# conf=0.5: 只保留置信度大于 0.5 的框
# iou=0.45: NMS 的 IoU 阈值
results = model(image_path, conf=0.5, iou=0.45)

# 解析结果
result = results[0]
boxes = result.boxes

print(f"检测到 {len(boxes)} 个目标")
print()

for i, box in enumerate(boxes, 1):
    x1, y1, x2, y2 = box.xyxy[0].tolist()
    conf = float(box.conf[0])
    cls = int(box.cls[0])
    name = model.names[cls]
    print(f"[{i}] {name:<12} 置信度: {conf:.3f}  坐标: [{x1:.1f}, {y1:.1f}, {x2:.1f}, {y2:.1f}]")


## 5. 可视化检测结果

YOLO 结果对象自带 plot 方法，可以方便地画出检测框。


In [ ]:
# 获取带框的图片（numpy 数组，BGR 格式）
annotated_img = result.plot()

# BGR 转 RGB 用于 matplotlib 显示
annotated_img_rgb = cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(12, 8))
plt.imshow(annotated_img_rgb)
plt.axis('off')
plt.title('YOLOv8 检测结果', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 6. 调整置信度阈值 conf

`conf` 控制"信不过就不框"：
- conf 低：检测更全，但可能包含更多误检
- conf 高：只保留最确定的框，可能漏检

下面我们在同一张图上尝试不同 conf 值。


In [ ]:
conf_values = [0.1, 0.3, 0.5, 0.7, 0.9]
fig, axes = plt.subplots(1, len(conf_values), figsize=(20, 4))

for ax, conf in zip(axes, conf_values):
    results = model(image_path, conf=conf, iou=0.45)
    annotated = results[0].plot()
    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

    ax.imshow(annotated_rgb)
    ax.set_title(f'conf={conf}', fontsize=11)
    ax.axis('off')

plt.suptitle('不同置信度阈值下的检测结果', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


### 观察

- conf=0.1：框很多，可能有一些错误的框
- conf=0.9：框很少，只保留最确定的

实际应用中：
- 需要高召回率（不漏掉目标）：用低 conf
- 需要高精度（减少误检）：用高 conf


## 7. 调整 NMS IoU 阈值

`iou` 控制 NMS 的严格程度：
- iou 低：更容易去掉重叠框，最终框更少
- iou 高：允许更多重叠框保留

这在密集目标检测中很重要，比如一群人站得很近。


In [ ]:
iou_values = [0.1, 0.3, 0.5, 0.7, 0.9]
fig, axes = plt.subplots(1, len(iou_values), figsize=(20, 4))

for ax, iou in zip(axes, iou_values):
    results = model(image_path, conf=0.3, iou=iou)
    annotated = results[0].plot()
    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

    ax.imshow(annotated_rgb)
    ax.set_title(f'iou={iou}', fontsize=11)
    ax.axis('off')

plt.suptitle('不同 NMS IoU 阈值下的检测结果', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 8. 批量图片推理

YOLO 支持直接对文件夹或图片列表做批量推理。


In [ ]:
# 创建一个临时图片列表
# 这里复用同一张图片多次作为示例
image_paths = [image_path] * 4 if image_path else [None] * 4

results = model(image_paths, conf=0.5, iou=0.45)

print(f"批量推理完成，共 {len(results)} 张图片")
for i, r in enumerate(results):
    print(f"  图片 {i+1}: 检测到 {len(r.boxes)} 个目标")


## 9. 多模型对比：速度与精度

YOLOv8 提供了不同大小的模型。我们对比 yolov8n、yolov8s、yolov8m 的速度和检测效果。

注意：yolov8s 和 yolov8m 需要下载更大的权重文件。


In [ ]:
import time

model_names = ['yolov8n.pt', 'yolov8s.pt']
comparison = []

for name in model_names:
    print(f"\n加载 {name} ...")
    m = YOLO(name)
    print(f"参数量: {sum(p.numel() for p in m.parameters()) / 1e6:.2f}M")

    #  warm up
    _ = m(image_path, conf=0.5, iou=0.45, verbose=False)

    # 测试推理时间
    start = time.time()
    results = m(image_path, conf=0.5, iou=0.45, verbose=False)
    elapsed = time.time() - start

    num_boxes = len(results[0].boxes)
    comparison.append({
        'model': name,
        'params': sum(p.numel() for p in m.parameters()) / 1e6,
        'time_ms': elapsed * 1000,
        'num_boxes': num_boxes
    })

    print(f"推理时间: {elapsed*1000:.1f} ms, 检测框数: {num_boxes}")

# 画对比图
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
models = [c['model'] for c in comparison]
times = [c['time_ms'] for c in comparison]
params = [c['params'] for c in comparison]

axes[0].bar(models, times, color=['#0d9f8f', '#d4a55f'])
axes[0].set_ylabel('推理时间 (ms)')
axes[0].set_title('速度对比')
for i, v in enumerate(times):
    axes[0].text(i, v + 5, f'{v:.1f}', ha='center')

axes[1].bar(models, params, color=['#0d9f8f', '#d4a55f'])
axes[1].set_ylabel('参数量 (M)')
axes[1].set_title('模型大小对比')
for i, v in enumerate(params):
    axes[1].text(i, v + 1, f'{v:.1f}', ha='center')

plt.tight_layout()
plt.show()


## 10. 在自定义图片上检测

如果你有本地图片，可以上传到这里并检测。


In [ ]:
# 尝试读取本地图片
# 把你想检测的图片放在当前目录，命名为 my_image.jpg
local_image = 'my_image.jpg'

if os.path.exists(local_image):
    results = model(local_image, conf=0.5, iou=0.45)
    annotated = results[0].plot()
    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(12, 8))
    plt.imshow(annotated_rgb)
    plt.axis('off')
    plt.title('自定义图片检测结果', fontsize=14, fontweight='bold')
    plt.show()
else:
    print(f"未找到 {local_image}，请把图片放到当前目录再运行")


## 11. YOLOv8 的网络改进（简要）

相比早期 YOLO 版本，YOLOv8 有几个重要改进：

1. **Anchor-free**：不再依赖预定义的 anchor 尺寸，模型直接预测框的中心和宽高
2. **解耦头（Decoupled Head）**：分类和回归（框坐标）分开预测，更稳定
3. **C2f 模块**：更高效的特征提取模块，兼顾速度和精度
4. **更强大的数据增强**： Mosaic、MixUp 等

这些改进让 YOLOv8 在速度和精度上都有很好的表现。


## 12. 动手练习

### 练习 1：找到最佳 conf 和 iou
对你的一张图片，尝试不同的 conf 和 iou 组合，找到你认为效果最好的参数。

### 练习 2：统计类别数量
对一张图片做检测，统计每个类别出现的次数。

### 练习 3：保存检测结果
把检测结果保存为图片文件，并输出每个框的坐标和类别到 txt 文件。

### 练习 4：视频检测（可选）
尝试用 YOLO 对视频文件或摄像头做实时检测：
```python
results = model('video.mp4', show=True)
```


## 13. 常见问题

**Q1：YOLO 能检测自定义类别吗？**
A：可以，但需要在自定义数据集上重新训练。本节课只讲推理，训练会在后续课程中涉及。

**Q2：conf 和 iou 怎么调？**
A：一般先固定 iou=0.45，调整 conf。如果密集目标多，可以适当提高 iou。

**Q3：YOLOv8 为什么这么快？**
A：它是 one-stage 检测器，一次前向传播同时完成分类和定位，不需要两阶段网络的区域提议过程。

**Q4：检测框的坐标格式是什么？**
A：`xyxy` 格式是 $(x_1, y_1, x_2, y_2)$，即左上角和右下角坐标。


## 14. 小结

- 目标检测 = 分类 + 定位，输出多个边界框
- YOLO 是 one-stage 检测器，速度快，适合实时应用
- 检测流程：特征提取 → 预测 → 置信度过滤 → NMS → 可视化
- conf 控制"信不信"，iou 控制"去重"
- YOLOv8n/s/m 是不同大小的模型，可以按需选择

下一节课我们将学习语义分割：不仅要找到目标，还要精确到每个像素。
